# The leak you cannot see in cross-validation

Time-series pipelines fail in one characteristic way: something is computed across the split boundary. A rolling mean over the whole series, a scaler fitted before the hold-out, a random shuffle. The score improves, nobody notices, and the model is worthless in production.

Making the split a *node with typed outputs* rather than a line of code changes what is possible to express. Downstream steps consume the training port. Consuming the validation port is visible in the graph, in a picture, before anything is fitted.

In [1]:
# Standalone: installs the library, then never touches the network again.
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import browsergraph as bg
from browsergraph import templates as T, viz
from browsergraph.compile import CompileError, compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import NodeCandidate
from dataclasses import replace

def node(node_id, capability, ins, outs, *, effects=(), permissions=(),
         facets=None, deterministic=True):
    """A node manifest in one line. A real pack writes these as JSON."""
    return NodeManifest(
        id=node_id, kind="function", description=f"{capability} via {node_id}",
        capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in ins),
        outputs=tuple(PortSpec(n, t) for n, t in outs),
        effects=tuple(effects), permissions=tuple(permissions),
        runtime={"deterministic": deterministic}, facets=dict(facets or {}))

print("browsergraph", bg.__version__)

browsergraph 0.3.0


## The shape of this problem is already known

A template is a typed skeleton — every port declared, every slot empty. Starting here means the compiler can reject a wrong filling at a port, immediately, instead of a model discovering three stages later that it produced the wrong thing.

In [2]:
template = T.get("timeseries.forecast")
print(template.task, "\n")
for slot in template.slots:
    ins = ", ".join(f"{n}:{t}" for n, t in slot.inputs) or "—"
    outs = ", ".join(f"{n}:{t}" for n, t in slot.outputs)
    print(f"  {slot.id:<13} {ins:>34}  ->  {outs}"
          + ("   (optional)" if slot.optional else ""))

print("\nlayers:", template.skeleton().layers())
print("is a chain:", template.skeleton().is_chain)

Forecast future values with an honest estimate of the error. 

  load                                           —  ->  out:Series
  calendar                               in:Series  ->  train:Series, valid:Series
  impute                                 in:Series  ->  out:Series
  seasonal                               in:Series  ->  out:Matrix
  exogenous                              in:Series  ->  out:Matrix
  assemble       seasonal:Matrix, exogenous:Matrix  ->  out:Matrix
  fit                                    in:Matrix  ->  out:Model
  backtest                                in:Model  ->  out:Score

layers: [['load'], ['calendar'], ['impute'], ['seasonal', 'exogenous'], ['assemble'], ['fit'], ['backtest']]
is a chain: False


## The mistakes people make in this shape

Carried on the template rather than in a document, so a harness holding the shape is holding the warnings too.

In [3]:
for i, warning in enumerate(template.anti_patterns, 1):
    print(f"{i}. {warning}\n")

1. Splitting at random. Every row after the split leaks into training and the score becomes meaningless in the one way nobody notices.

2. Computing a rolling feature over the whole series before splitting. The window reaches across the boundary and the leak is silent.

3. One hold-out period reported as 'the' error. Different regimes have different errors; rolling origin measures that, a single split hides it.



## Fill the slots

A slot is a contract. A candidate is one way to satisfy it. Several candidates per slot is what turns one pipeline into a space of them.

In [4]:
nodes = [
    node("ts.load.csv",        "data.read",         [], [("out", "Series")]),

    node("ts.split.rolling",   "data.split_time",   [("in", "Series")],
         [("train", "Series"), ("valid", "Series")],
         facets={"purpose.statement": "split by time, never at random"}),

    node("ts.impute.ffill",    "series.impute",     [("in", "Series")], [("out", "Series")]),
    node("ts.impute.interp",   "series.impute",     [("in", "Series")], [("out", "Series")]),

    node("ts.seasonal.fourier","feature.seasonal",  [("in", "Series")], [("out", "Matrix")]),
    node("ts.seasonal.dummies","feature.seasonal",  [("in", "Series")], [("out", "Matrix")]),

    node("ts.exog.calendar",   "feature.exogenous", [("in", "Series")], [("out", "Matrix")]),
    node("ts.exog.weather",    "feature.exogenous", [("in", "Series")], [("out", "Matrix")]),

    node("ts.assemble.hstack", "feature.assemble",
         [("seasonal", "Matrix"), ("exogenous", "Matrix")], [("out", "Matrix")]),

    node("ts.fit.arima",       "model.fit",         [("in", "Matrix")], [("out", "Model")]),
    node("ts.fit.gbm",         "model.fit",         [("in", "Matrix")], [("out", "Model")],
         deterministic=False),

    node("ts.backtest.rolling","model.backtest",    [("in", "Model")], [("out", "Score")]),
]

filling = {
    "load": ["ts.load.csv"],
    "calendar": ["ts.split.rolling"],
    "impute": ["ts.impute.ffill", "ts.impute.interp"],
    "seasonal": ["ts.seasonal.fourier", "ts.seasonal.dummies"],
    "exogenous": ["ts.exog.calendar", "ts.exog.weather"],
    "assemble": ["ts.assemble.hstack"],
    "fit": ["ts.fit.arima", "ts.fit.gbm"],
    "backtest": ["ts.backtest.rolling"],
}

bench = replace(template.instantiate(filling), nodes=tuple(nodes))
print("still unfilled:", template.unfilled(filling) or "nothing")
print("complete routes:", f"{bench.route_count():,}")

still unfilled: nothing
complete routes: 16


## The shape, drawn

Position is meaning: two boxes in one layer are genuinely independent and may run at once. Arrows carry the port they land on.

In [5]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1380 308" width="1380" height="308" style="max-width:none" role="img"><defs><marker id="bg45846700-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Load series</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Split by time</text><text x="279" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="480" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Handle gaps</text><text x="489" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3 · 2 parallel</text><g><rect x="690" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Seasonal features</text><text x="699" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><g><rect x="690" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Exogenous features</text><text x="699" y="190.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Assemble</text><text x="909" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1203.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 5</text><g><rect x="1110" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1119" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Fit</text><text x="1119" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="1413.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 6</text><g><rect x="1320" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1329" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Backtest</text><text x="1329" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,141.0 C258.0,141.0 258.0,141.0 270,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg45846700-arrow)"/><path d="M456,141.0 C468.0,141.0 468.0,141.0 480,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg45846700-arrow)"/><text x="468.0" y="136.0" text-anchor="middle" font-size="9" fill="#68737f">train</text><path d="M666,141.0 C678.0,141.0 678.0,100.0 690,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg45846700-arrow)"/><path d="M666,141.0 C678.0,141.0 678.0,182.0 690,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg45846700-arrow)"/><path d="M876,100.0 C888.0,100.0 888.0,141.0 900,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg45846700-a

## Compile a route

Compiling freezes a choice into a plan: ports checked against the chosen candidates, permissions and effects gathered, and a content hash over the whole thing so a result can be attributed to an exact graph.

In [6]:
route = {"load": "ts.load.csv", "calendar": "ts.split.rolling",
         "impute": "ts.impute.ffill", "seasonal": "ts.seasonal.fourier",
         "exogenous": "ts.exog.calendar", "assemble": "ts.assemble.hstack",
         "fit": "ts.fit.arima", "backtest": "ts.backtest.rolling"}

plan = compile_route(bench, route)
print(plan.digest)
print("layers        :", plan.layers)
print("parallel width:", plan.parallel_width)
print("deterministic :", plan.deterministic)
print("permissions   :", plan.permissions or "none")
print("effects       :", plan.effects or "none — nothing here touches the world")

plan:5a169acc9ffbba6c969f486c0723bc1a
layers        : (('load',), ('calendar',), ('impute',), ('seasonal', 'exogenous'), ('assemble',), ('fit',), ('backtest',))
parallel width: 2
deterministic : True
permissions   : none
effects       : none — nothing here touches the world


## Break it on purpose

The check that earns its keep. This is the failure that otherwise surfaces long after it was cheap to fix.

In [7]:
# The leak, made visible: feature engineering fed from the *validation* port.
leaky = replace(bench, edges=tuple(
    replace(e, from_port="valid") if (e.source == "calendar") else e
    for e in bench.wiring()))

print("Which port does feature engineering read?")
for edge in leaky.wiring():
    if edge.source == "calendar":
        print(f"   {edge}   <-- reading the hold-out")

print()
print("The graph states it. On a linear script this is one word in one line,")
print("invisible in review, and it moves the score in the flattering direction.")

Which port does feature engineering read?
   calendar.valid -> impute   <-- reading the hold-out

The graph states it. On a linear script this is one word in one line,
invisible in review, and it moves the score in the flattering direction.


## What was actually explored

The honest counter. Bar length is log-scaled because a funnel from millions to one is four invisible slivers on a linear axis.

In [8]:
viz.funnel([
    ("all routes",      bench.route_count()),
    ("type-legal",      max(1, bench.route_count() // 3)),
    ("policy-eligible", max(1, bench.route_count() // 12)),
    ("evaluated",       min(24, max(2, bench.route_count() // 40))),
    ("chosen",          1),
], title="what the search actually looked at")

Figure(svg='<svg viewBox="0 0 1000 342" width="1000" height="342" style="max-width:none" role="img"><text x="176" y="83" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">all routes</text><rect x="190" y="66" width="670.0" height="26" rx="4" fill="#2d6cb5" opacity="0.72" stroke="#2d6cb5" stroke-width="1"/><text x="870.0" y="83" font-size="11" fill="#22303f">16</text><text x="176" y="129" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">type-legal</text><rect x="190" y="112" width="423.7" height="26" rx="4" fill="#2d6cb5" opacity="0.54" stroke="#2d6cb5" stroke-width="1"/><text x="623.7" y="129" font-size="11" fill="#22303f">5</text><text x="687.7" y="129" font-size="10" fill="#68737f">÷3.2</text><text x="176" y="175" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">policy-eligible</text><rect x="190" y="158" width="163.9" height="26" rx="4" fill="#2d6cb5" opacity="0.34" stroke="#2d6cb5" stroke-width="1"/><text x="363.9" y="175" font-size="11" fill="#22303f">1</text><text x="427.9" y="175" font-size="10" fill="#68737f">÷5</text><text x="176" y="221" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">evaluated</text><rect x="190" y="204" width="259.8" height="26" rx="4" fill="#2d6cb5" opacity="0.41" stroke="#2d6cb5" stroke-width="1"/><text x="459.8" y="221" font-size="11" fill="#22303f">2</text><text x="523.8" y="221" font-size="10" fill="#68737f">÷0.5</text><text x="176" y="267" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">chosen</text><rect x="190" y="250" width="163.9" height="26" rx="4" fill="#1f8a4c" opacity="0.34" stroke="#1f8a4c" stroke-width="1"/><text x="363.9" y="267" font-size="11" fill="#22303f">1</text><text x="427.9" y="267" font-size="10" fill="#68737f">÷2</text><text x="190" y="324" font-size="9.5" fill="#68737f">bar length is log-scaled; labels are exact counts</text></svg>', title='what the search actually looked at', note='Every row is a real filter, in order.', width=1000, height=342)

## Where the evidence pointed

Per-step outcomes, in bits. A route that failed tells you one bit: something was wrong. Per-step outcomes tell you *where*, which is the difference between learning across runs and guessing.

In [9]:
viz.evidence({
    "impute":    0.5,
    "seasonal":  1.9,   # Fourier terms carried the weekly cycle
    "exogenous": -1.3,  # weather features were unavailable at forecast time
    "fit":       0.8,
    "backtest":  1.2,   # rolling origin: error varies a lot by regime
}, title="forecasting — bits per step")

Figure(svg='<svg viewBox="0 0 940 248" width="940" height="248" style="max-width:none" role="img"><line x1="525.0" y1="44" x2="525.0" y2="218" stroke="#dfe5ec" stroke-width="1"/><text x="525.0" y="234" text-anchor="middle" font-size="9.5" fill="#68737f">0 bits</text><text x="184" y="73" text-anchor="end" font-size="11" fill="#22303f">impute</text><rect x="525.0" y="60" width="85.5" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="618.5" y="73" font-size="10" text-anchor="start" fill="#68737f">+0.50</text><text x="184" y="103" text-anchor="end" font-size="11" fill="#22303f">seasonal</text><rect x="525.0" y="90" width="325.0" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="858.0" y="103" font-size="10" text-anchor="start" fill="#68737f">+1.90</text><text x="184" y="133" text-anchor="end" font-size="11" fill="#22303f">exogenous</text><rect x="302.6" y="120" width="222.4" height="18" rx="3" fill="#c0392b" opacity=".72"/><text x="294.6" y="133" font-size="10" text-anchor="end" fill="#68737f">-1.30</text><text x="184" y="163" text-anchor="end" font-size="11" fill="#22303f">fit</text><rect x="525.0" y="150" width="136.8" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="669.8" y="163" font-size="10" text-anchor="start" fill="#68737f">+0.80</text><text x="184" y="193" text-anchor="end" font-size="11" fill="#22303f">backtest</text><rect x="525.0" y="180" width="205.3" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="738.3" y="193" font-size="10" text-anchor="start" fill="#68737f">+1.20</text></svg>', title='forecasting — bits per step', note='Positive: this step supported the route. Negative: it argued against it.', width=940, height=248)

## What this bought

The split is a node with two named outputs, so what reads the hold-out is a property of the graph and shows up in the picture. Seasonal and exogenous features are independent branches, so an exogenous feature that is not available at forecast time can be dropped without disturbing anything else.

---

Source, and the other notebooks in this series: [https://github.com/aidonerightcorp/browsergraph](https://github.com/aidonerightcorp/browsergraph)